# MAGPIE — May 2026 Demo

Run cells top to bottom. **Cell 1** starts all nodes. **Cell 2** makes rclpy importable in VS Code.

Set `OBJECT` in Cell 3 to the thing on the table, e.g. `'red block'`.

In [ ]:
import subprocess, os, time

SAM3_PY  = os.path.expanduser('~/sam3_env/bin/python3')
SAM3_SCR = os.path.expanduser('~/magpie_control/scripts/sam3_infer.py')
ROS      = 'source /opt/ros/humble/setup.bash && source ~/ws_ctrl/install/setup.bash'

def ros_proc(cmd, log):
    open(log, 'w').close()
    return subprocess.Popen(f'{ROS} && {cmd}', shell=True, executable='/bin/bash',
                            stdout=open(log, 'a'), stderr=subprocess.STDOUT)

print('Killing stale processes...')
_procs = ['ur5_node', 'gripper_node', 'realsense2_camera', 'sam3_infer', 'slip_guard_node', 'ft_sensor_node', 'graspgenx_server']
for p in _procs:
    subprocess.run(f'pkill -f {p}', shell=True)      # SIGTERM: ur5_node catches it and releases its RTDE script cleanly
time.sleep(2.5)                                       # let nodes shut down gracefully
for p in _procs:
    subprocess.run(f'pkill -9 -f {p}', shell=True)    # force-kill any stragglers that ignored SIGTERM
subprocess.run('rm -f /tmp/sam3.sock', shell=True)
time.sleep(3)

PROCS = {}
PROCS['ur5']     = ros_proc('ros2 run magpie_control ur5_node',     '/tmp/log_ur5.txt')
PROCS['gripper'] = ros_proc('ros2 run magpie_control gripper_node', '/tmp/log_gripper.txt')
PROCS['ft']      = ros_proc('ros2 run magpie_control ft_sensor_node', '/tmp/log_ft.txt')
PROCS['camera']  = ros_proc(
    'ros2 run realsense2_camera realsense2_camera_node --ros-args -r __ns:=/camera/gripper_camera',
    '/tmp/log_camera.txt')
PROCS['slip_guard'] = ros_proc(
    f'python3 {os.path.expanduser("~/magpie_control/scripts/slip_guard_node.py")}',
    '/tmp/log_slip_guard.txt')
PROCS['sam3']    = subprocess.Popen(
    [SAM3_PY, SAM3_SCR, '--socket'],
    stdout=open('/tmp/log_sam3.txt', 'w'), stderr=subprocess.STDOUT)

# ── Grasp memory (force priors + DINO RAG) ──────────────────────────
import sys as _sys
_sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts'))
from grasp_memory import GraspMemory
gm = GraspMemory(os.path.expanduser('~/magpie_control/data/grasp_log'))
print(f'GraspMemory loaded — {len(gm._priors)} known objects')

print('Waiting for SAM3 (up to 90s)...')
for i in range(90):
    time.sleep(1)
    if os.path.exists('/tmp/sam3.sock'):
        print(f'  SAM3 ready after {i+1}s')
        break
    if PROCS['sam3'].poll() is not None:
        print('  SAM3 crashed:'); print(open('/tmp/log_sam3.txt').read()[-600:])
        break
    if (i+1) % 20 == 0:
        print(f'  {i+1}s...')

time.sleep(3)
print()
# Auto-retry ur5 if it didn't come up — the dashboard-clear in ur5.start() means a retry
# almost always succeeds WITHOUT a pendant controller restart.
for _try in range(2):
    if PROCS['ur5'].poll() is None: break
    print(f'  ur5 EXITED — auto-retrying ({_try+1}/2, self-clearing stuck script)...')
    subprocess.run('pkill -9 -f ur5_node', shell=True); time.sleep(2)
    PROCS['ur5'] = ros_proc('ros2 run magpie_control ur5_node', '/tmp/log_ur5.txt')
    time.sleep(6)

# ── Preload GraspGenX server in background ────────────────────────────────────
# GraspGenX (~0.8 GB) coexists with SAM3 (~3.9 GB) on the 8 GB card — measured
# ~3 GB headroom. Start the ZMQ server now (loads in ~10s) so it's warm for the
# first grasp and stays warm all session (no per-grasp swap).
_ggx_sh = os.path.expanduser('~/magpie_control/scripts/run_graspgenx_server.sh')
if os.path.exists(_ggx_sh):
    PROCS['graspgenx'] = subprocess.Popen(
        ['bash', _ggx_sh],
        stdout=open('/tmp/log_graspgenx.txt', 'w'), stderr=subprocess.STDOUT)
    print('  graspgenx  STARTING (loading model in background, ~60s — see /tmp/log_graspgenx.txt)')
else:
    print('  graspgenx  script not found — will auto-start if GRASP_METHOD=graspgenx')
for n, p in PROCS.items():

    status = 'OK' if p.poll() is None else f'EXITED  -> tail /tmp/log_{n}.txt'
    print(f'  {n:<10} {status}')

In [ ]:
# ── Memory Manager ────────────────────────────────────────────────────────────
# Run this to inspect, rename, or delete entries from grasp memory.
#
# Override auto-detection for next pickup:
#   ACTIVE_OVERRIDE = 'measuring tape'
# Clear it after:
#   ACTIVE_OVERRIDE = ''

ACTIVE_OVERRIDE = ''   # ← set this before running 17ea9af9 if detection is wrong

print('=== Grasp Memory Contents ===')
if 'gm' not in dir():
    import sys; sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts'))
    from grasp_memory import GraspMemory
    gm = GraspMemory(os.path.expanduser('~/magpie_control/data/grasp_log'))
for _name, _p in sorted(gm._priors.items(), key=lambda x: -x[1]['n']):
    print(f'  {_name:<30} n={_p["n"]:>3}  force={_p["x"]:.1f}N  std={_p["P"]**0.5:.2f}N')
print(f'\nTotal: {len(gm._priors)} objects')

# ── To delete a wrong entry ────────────────────────────────────────────────
# del gm._priors['wrong name']; gm._save_priors(); print('Deleted.')

# ── To rename a wrong entry ───────────────────────────────────────────────
# gm._priors['correct name'] = gm._priors.pop('wrong name')
# gm._save_priors(); print('Renamed.')

# ── To reset a single object to re-learn from scratch ─────────────────────
# del gm._priors['measuring tape']; gm._save_priors(); print('Reset.')


In [ ]:
# ═══ GRASP-METHOD COMPARISON (uses the REAL pickup) ═══════════════════════════
# Run this once. Then per method:  set GRASP_METHOD, run the pickup cell (full run),
# reset the object to the same spot, repeat. Finally run cmp_table.
CMP_RESULTS = []
GRASP_METHOD = 'graspgenx'  # 'pca' | 'graspgen' | 'graspgenx'   (graspgenx = MAGPIE gripper, NVlabs GraspGenX)
VLA_RERUN    = 'spawn'      # opens live Rerun window — set None to disable
if 'vla_rec' in dir(): del vla_rec  # force recreation with new VLA_RERUN setting
print(f'comparison armed — GRASP_METHOD={GRASP_METHOD!r}, CMP_RESULTS reset. '
      'Set GRASP_METHOD then run the pickup.')

In [ ]:
# Load ROS shared libraries so rclpy imports in any VS Code kernel
import sys, os, ctypes, glob

for _d in [
    '/opt/ros/humble/lib/x86_64-linux-gnu',
    '/opt/ros/humble/lib',
    '/home/user/ws_ctrl/install/magpie_msgs/lib',
    '/home/user/ws_ctrl/install/magpie_control/lib',
]:
    for _so in sorted(glob.glob(_d + '/*.so*')):
        try:
            ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

for _p in [
    '/opt/ros/humble/local/lib/python3.10/dist-packages',
    '/opt/ros/humble/lib/python3.10/site-packages',
    '/home/user/ws_ctrl/install/magpie_msgs/local/lib/python3.10/dist-packages',
    '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages',
    '/home/user/.local/lib/python3.10/site-packages',
    '/home/user/magpie_control/src',
    '/home/user/magpie_control/scripts',
]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import rclpy
print('rclpy OK')

In [ ]:
import time, re, json, tempfile, base64, socket as _sock
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import cv2
%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['figure.dpi']     = 80   # keeps output file size small for GitHub

from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy
from sensor_msgs.msg import Image as RosImage, CameraInfo
from geometry_msgs.msg import PoseStamped, Pose, WrenchStamped
from std_srvs.srv import Trigger
from cv_bridge import CvBridge
from magpie_msgs.msg import GripperState
from magpie_msgs.srv import MoveLinear, SetGripperForce, SetGripperPosition
from magpie_control import poses
from magpie_control.homog_utils import homog_xform, R_krot
from magpie_control.gripper_arc import fingertip_drop
from google import genai
from google.genai import types as gtypes
from pointcloud_utils import build_segmented_pcd, denoise_pcd, analyse_pcd

# --- CONFIG ---
OBJECT         = ''   # blank = Gemini auto-detects
SAM3_SOCK      = '/tmp/sam3.sock'
GEMINI_KEY     = os.environ.get('GEMINI_API_KEY', '')
GRIPPER_LEN    = 0.231
HARD_FLOOR_Z   = 0.060   # overridden by TABLE_Z cell → always tracks real table surface
TABLE_Z        = None   # set by running the measure cell below
CAMERA_MOUNT_Z_OFFSET_M = 0.024  # extend OBB aug bottom below table surface; z_off in _TCP_TO_CAM already accounts for mount height
APPROACH_H     = 0.10
# Camera extrinsic: transform from camera optical frame to TCP frame.
# z_off=0.144 m = physical camera-to-TCP distance (0.120 + 0.024 measured mount error).
# Rz(-90°) is the last-working clocking — run the calibration cell (cell 7) at setup
# to re-measure the correct clocking and confirm z_off for this mount.
# The calibration cell overwrites _TCP_TO_CAM for the session.
# XY offset of camera center relative to gripper jaw center in tool frame (metres).
# Tune by observing consistent grasp miss direction: if arm always misses +5mm in world-X,
# add that error (in tool frame) here.  Positive X = camera is offset toward robot.
_CAM_XY_OFFSET_M = (0.0, 0.0)  # (x_tool_m, y_tool_m) — adjust to compensate lateral camera offset
_TCP_TO_CAM    = homog_xform(R_krot([0, 0, 1], -np.pi/2), [_CAM_XY_OFFSET_M[0], _CAM_XY_OFFSET_M[1], 0.144])  # z_off updated: 0.120+0.024 mount

# sam3_query defined HERE (early) so every downstream cell has it after a kernel reload
# — no more 'sam3_query is not defined' from running cells out of order.
def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp = tempfile.mktemp(suffix='.jpg')
    cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
            s.connect(sock_path)
            s.sendall((json.dumps({'image': tmp, 'query': query}) + '\n').encode())
            raw = b''
            while True:
                chunk = s.recv(65536)
                if not chunk: break
                raw += chunk
        d = json.loads(raw.decode().strip())
        if 'error' in d: raise RuntimeError(d['error'])
        boxes  = np.array(d['boxes'],  dtype=float)
        scores = np.array(d['scores'], dtype=float)
        mask   = None
        if d.get('mask_b64') and len(boxes) > 0:
            raw2 = base64.b64decode(d['mask_b64'])
            h, w = d['mask_shape']
            mask = np.frombuffer(raw2, dtype=np.uint8).reshape(h, w).astype(bool)
        return boxes, scores, mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

if not GEMINI_KEY:
    print('WARNING: GEMINI_API_KEY not set.')
print('Imports OK')

In [ ]:
# Helper: quaternion <-> rotation vector
def _quat_to_rv(w, x, y, z):
    a = 2.0 * np.arccos(np.clip(w, -1, 1))
    s = np.sin(a / 2)
    return np.zeros(3) if s < 1e-10 else a * np.array([x, y, z]) / s

def _rv_to_quat(rv):
    a = np.linalg.norm(rv)
    if a < 1e-10:
        return (1., 0., 0., 0.)
    ax = rv / a
    return (np.cos(a/2), ax[0]*np.sin(a/2), ax[1]*np.sin(a/2), ax[2]*np.sin(a/2))

def _mat_to_pose(mat):
    v = poses.pose_mtrx_to_vec(np.array(mat))
    w, x, y, z = _rv_to_quat(np.array(v[3:]))
    p = Pose()
    p.position.x, p.position.y, p.position.z = v[0], v[1], v[2]
    p.orientation.w, p.orientation.x = w, x
    p.orientation.y, p.orientation.z = y, z
    return p

# QoS profiles matching RealSense D405 publisher settings
img_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.TRANSIENT_LOCAL)
inf_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.VOLATILE)

class Demo(Node):
    def __init__(self):
        super().__init__('magpie_demo')
        self.bridge = CvBridge()
        self.color = self.depth = self.caminfo = self.tcp = self.gs = None
        self.wrench = None   # WrenchStamped from OptoForce FT (may be None)
        NS = '/camera/gripper_camera/camera'
        self.create_subscription(
            RosImage,    NS+'/color/image_raw',
            lambda m: setattr(self, 'color', self.bridge.imgmsg_to_cv2(m, 'rgb8')), img_qos)
        self.create_subscription(
            RosImage,    NS+'/depth/image_rect_raw',
            lambda m: setattr(self, 'depth', self.bridge.imgmsg_to_cv2(m, 'passthrough')), img_qos)
        self.create_subscription(
            CameraInfo,  NS+'/color/camera_info',
            lambda m: setattr(self, 'caminfo', m), inf_qos)
        self.create_subscription(
            GripperState, '/gripper/state',
            lambda m: setattr(self, 'gs', m), 1)
        self.create_subscription(
            WrenchStamped, 'ft_sensor/wrench',
            lambda m: setattr(self, 'wrench', m), 10)
        self.create_subscription(
            PoseStamped, '/arm/tcp_pose',
            self._tcp_cb, 1)
        self.mv  = self.create_client(MoveLinear,         '/arm/move_l')
        self.tch = self.create_client(Trigger,             '/arm/teach_mode')
        self.opn = self.create_client(Trigger,             '/gripper/open')
        self.cls = self.create_client(Trigger,             '/gripper/close')
        self.frc = self.create_client(SetGripperForce,     '/gripper/set_force')
        self.pos = self.create_client(SetGripperPosition,  '/gripper/set_position')
        self.clr = self.create_client(Trigger,             '/gripper/clear_error')

    def _tcp_cb(self, m):
        p = m.pose
        rv = _quat_to_rv(p.orientation.w, p.orientation.x, p.orientation.y, p.orientation.z)
        self.tcp = poses.pose_vec_to_mtrx([p.position.x, p.position.y, p.position.z, *rv])

    def spin(self, n=10, t=0.15):
        for _ in range(n):
            rclpy.spin_once(self, timeout_sec=t)

    def _call(self, client, req, timeout=30.):
        client.wait_for_service(timeout_sec=4.)
        fut = client.call_async(req)
        rclpy.spin_until_future_complete(self, fut, timeout_sec=timeout)
        return fut.result()

    def move(self, mat, spd=0.08, acc=0.2):
        r = MoveLinear.Request()
        r.target_pose = _mat_to_pose(mat)
        r.speed = spd; r.acceleration = acc; r.async_mode = False
        resp = self._call(self.mv, r)
        if not resp.success:
            raise RuntimeError(resp.message)

    def open_g(self):  return self._call(self.opn, Trigger.Request())
    def close_g(self): return self._call(self.cls, Trigger.Request())

    def set_force(self, n):
        r = SetGripperForce.Request(); r.max_force = float(n)
        return self._call(self.frc, r)

    def set_pos(self, mm):
        r = SetGripperPosition.Request(); r.position = float(max(0, mm))
        return self._call(self.pos, r)

    def clear_err(self):
        # Re-enable AX-12 torque after overload shutdown (reset_packet_overload).
        # NOTE: this resets force limit to 2N, so call set_force() again after.
        if not self.clr.wait_for_service(timeout_sec=1.):
            return None
        return self._call(self.clr, Trigger.Request())

    def slip_guard_enable(self, force_n, slip_thresh, obj_u=None, obj_v=None, force_step=1.0):
        """Send config (force, thresh, object pixel coords, reclamp step) then enable the SlipGuardNode."""
        if not hasattr(self, '_sg_en'):
            from std_msgs.msg import Float32MultiArray
            self._sg_cfg_pub = self.create_publisher(
                Float32MultiArray, 'slip_guard/config', 1)
            self._sg_en  = self.create_client(Trigger, 'slip_guard/enable')
            self._sg_dis = self.create_client(Trigger, 'slip_guard/disable')
            from std_msgs.msg import String as _SgStr
            self._sg_events = []
            self.create_subscription(_SgStr, '/slip_guard/events',
                lambda msg: self._sg_events.append(msg.data), 10)
        from std_msgs.msg import Float32MultiArray
        cfg = Float32MultiArray()
        # [force, thresh, obj_u, obj_v] — pixel coords tell guard where to sample depth
        cfg.data = [float(force_n), float(slip_thresh),
                    float(obj_u) if obj_u is not None else -1.,
                    float(obj_v) if obj_v is not None else -1.,
                    float(force_step)]
        self._sg_events = []  # reset for this grasp — fresh DAgger log
        self._sg_cfg_pub.publish(cfg)
        time.sleep(0.1)
        if not self._sg_en.wait_for_service(timeout_sec=1.):
            print('[slip_guard] node not running — guard skipped')
            return
        self._call(self._sg_en, Trigger.Request())

    def slip_guard_disable(self):
        if hasattr(self, '_sg_dis') and self._sg_dis.wait_for_service(timeout_sec=0.5):
            self._call(self._sg_dis, Trigger.Request())

    def unteach(self):
        if not self.tch.wait_for_service(timeout_sec=2.):
            return
        r = self._call(self.tch, Trigger.Request())
        if r and 'enabled' in r.message.lower():
            self._call(self.tch, Trigger.Request())

    def wait_sensors(self, timeout=20.):
        t0 = time.time()
        while time.time() - t0 < timeout:
            rclpy.spin_once(self, timeout_sec=0.15)
            if all(v is not None for v in [self.color, self.depth, self.caminfo, self.tcp]):
                return True
        return False

try:
    rclpy.init()
except RuntimeError:
    pass

try:
    node.destroy_node()
except Exception:
    pass

node = Demo()
ok   = node.wait_sensors(20.)
print('Sensors ready:', ok,
      '| color:', node.color is not None,
      '| depth:', node.depth is not None,
      '| tcp:',   node.tcp   is not None)

In [ ]:
import importlib, pointcloud_utils; importlib.reload(pointcloud_utils)
from pointcloud_utils import (build_segmented_pcd, denoise_pcd, analyse_pcd,
                              grasp_rotation_matrix, check_view_quality, top_layer,
                              smart_grasp_angle, augment_cloud_for_graspgenx,
                              rank_grasp_angles_visual)
import open3d as o3d

print('=== Full Pickup ===')
# ── Setup: auto-load calibration + auto-measure TABLE_Z ──────────────────────
import pathlib as _pp, datetime as _dt
_calib_path = _pp.Path('~/magpie_control/data/last_calibration.json').expanduser()
if abs(_TCP_TO_CAM[2, 3] - 0.144) > 0.05 and _calib_path.exists():
    import json as _jj
    _c = _jj.load(open(_calib_path))
    _age = (_dt.datetime.now() - _dt.datetime.fromisoformat(_c['saved_at'])).days
    from magpie_control.homog_utils import homog_xform, R_krot
    _TCP_TO_CAM = homog_xform(R_krot([0, 0, 1], np.radians(_c['clocking_deg'])),
                              [_c['cam_xy_offset'][0], _c['cam_xy_offset'][1], _c['z_off']])
    _CAM_XY_OFFSET_M = tuple(_c['cam_xy_offset'])
    if _age > 7:
        print(f'  [calib] loaded {_age}d-old calibration — consider re-running cell 7')
    else:
        print(f'  [calib] auto-loaded Rz({_c["clocking_deg"]}°) from {_age}d ago')
elif abs(_TCP_TO_CAM[2, 3] - 0.144) > 0.05:
    print('  [!] No saved calibration — using default _TCP_TO_CAM. Run cell 7 first.')
else:
    print('  [calib] calibration active this session ✓')


# Ensure dependencies exist if cells 8b/8c weren't run
if 'gc' not in dir():
    from google import genai
    gc = genai.Client(api_key=os.environ.get('GEMINI_API_KEY', ''))

if 'sam3_query' not in dir():
    import socket as _sock, json, base64, tempfile
    def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
        tmp = tempfile.mktemp(suffix='.jpg')
        cv2.imwrite(tmp, cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        try:
            with _sock.socket(_sock.AF_UNIX, _sock.SOCK_STREAM) as s:
                s.connect(sock_path)
                s.sendall((json.dumps({'image': tmp, 'query': query}) + '\n').encode())
                raw = b''
                while True:
                    chunk = s.recv(65536)
                    if not chunk: break
                    raw += chunk
            d = json.loads(raw.decode().strip())
            if 'error' in d: raise RuntimeError(d['error'])
            boxes  = np.array(d['boxes'],  dtype=float)
            scores = np.array(d['scores'], dtype=float)
            mask   = None
            if d.get('mask_b64') and len(boxes) > 0:
                raw2 = base64.b64decode(d['mask_b64'])
                h, w = d['mask_shape']
                mask = np.frombuffer(raw2, dtype=np.uint8).reshape(h, w).astype(bool)
            return boxes, scores, mask
        finally:
            if os.path.exists(tmp): os.unlink(tmp)

# ── 1. Snapshot + Gemini detect ───────────────────────────────────────────────
node.spin(8)
fp_color  = node.color.copy()
fp_depth  = node.depth.copy()
fp_tcp    = node.tcp.copy()
fp_ci     = node.caminfo
home_pick = fp_tcp.copy()

_, _buf = cv2.imencode('.jpg', cv2.cvtColor(fp_color, cv2.COLOR_RGB2BGR))
r_auto = gc.models.generate_content(
    model='gemini-2.5-flash',
    contents=[
        gtypes.Part.from_bytes(data=_buf.tobytes(), mime_type='image/jpeg'),
        'What is the main graspable object? Reply 2-4 words only, no punctuation.'
    ])
ACTIVE = ' '.join(r_auto.text.strip().lower().split()[:4])
if 'ACTIVE_OVERRIDE' in dir() and ACTIVE_OVERRIDE.strip():
    print(f'Gemini detected: "{ACTIVE}" → overridden to "{ACTIVE_OVERRIDE.strip()}"')
    ACTIVE = ACTIVE_OVERRIDE.strip().lower()
else:
    print(f'Gemini detected: "{ACTIVE}"  (set ACTIVE_OVERRIDE="name" to change)')

# ── 2. SAM3 with close-up retry ──────────────────────────────────────────────
MAX_DET_RETRIES = 3
CLOSE_STEP_M    = 0.02
_det_tcp = fp_tcp.copy()
fp_boxes = fp_scores = fp_mask = None

for _det_try in range(1 + MAX_DET_RETRIES):
    fp_boxes, fp_scores, fp_mask = sam3_query(fp_color, ACTIVE)
    if len(fp_scores) > 0 and fp_mask is not None:
        best_i = int(np.argmax(fp_scores))
        print(f'SAM3:   score={fp_scores[best_i]:.3f}  mask_px={fp_mask.sum()}'
              + (f'  (moved {_det_try*CLOSE_STEP_M*100:.0f}cm closer)' if _det_try else ''))
        break
    if _det_try < MAX_DET_RETRIES:
        print(f'  SAM3 try {_det_try+1}: no detection — moving 2cm closer...')
        _det_tcp = _det_tcp.copy(); _det_tcp[2, 3] -= CLOSE_STEP_M
        node.unteach(); node.move(_det_tcp, spd=0.05); node.spin(3)
        fp_color = node.color.copy(); fp_depth = node.depth.copy(); fp_tcp = _det_tcp
else:
    raise RuntimeError(f'SAM3: no detection for "{ACTIVE}" after {MAX_DET_RETRIES+1} tries')

# ── Auto TABLE_Z from ring around detected object ─────────────────────────────
# Dilate the SAM3 mask and sample depth from the ring outside it — pure table.
# This replaces the separate clear-table TABLE_Z cell.
if TABLE_Z is None:
    import cv2 as _cv2a
    _dil = _cv2a.dilate(fp_mask.astype(np.uint8), np.ones((50,50), np.uint8)).astype(bool)
    _ring = _dil & ~fp_mask
    _ys_r, _xs_r = np.where(_ring)
    _ds_r = fp_depth[_ring].astype(float)
    _ok_r = _ds_r > 0
    if _ok_r.sum() > 100:
        _xs_r, _ys_r, _ds_r = _xs_r[_ok_r], _ys_r[_ok_r], _ds_r[_ok_r] / 1000.
        _fx_r, _fy_r = fp_ci.k[0], fp_ci.k[4]
        _cx_r, _cy_r = fp_ci.k[2], fp_ci.k[5]
        _Xc = (_xs_r - _cx_r) * _ds_r / _fx_r
        _Yc = (_ys_r - _cy_r) * _ds_r / _fy_r
        _T_r = fp_tcp @ _TCP_TO_CAM
        _Zw = _T_r[2,0]*_Xc + _T_r[2,1]*_Yc + _T_r[2,2]*_ds_r + _T_r[2,3]
        TABLE_Z = float(np.median(_Zw))
        HARD_FLOOR_Z = TABLE_Z
        print(f'  [auto] TABLE_Z={TABLE_Z*1000:.1f}mm  HARD_FLOOR_Z={HARD_FLOOR_Z*1000:.1f}mm')
    else:
        raise RuntimeError('Auto TABLE_Z failed — not enough table visible around object. '
                           'Move arm closer or run cell 8 on a clear table.')
else:
    print(f'  [TABLE_Z] using pre-measured {TABLE_Z*1000:.1f}mm')

# ── Show: detection ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
vis0 = fp_color.copy()
vis0[fp_mask] = (vis0[fp_mask]*.4 + np.array([0,200,0])*.6).astype(np.uint8)
x1b,y1b,x2b,y2b = fp_boxes[best_i].astype(int)
# Save detection crop NOW — fp_color will be overwritten with scan 2 image later
_det_crop = fp_color[
    max(0,y1b):min(fp_color.shape[0],y2b),
    max(0,x1b):min(fp_color.shape[1],x2b)]
ax1.imshow(vis0)
ax1.add_patch(patches.Rectangle((x1b,y1b),x2b-x1b,y2b-y1b,lw=3,ec='lime',fc='none'))
ax1.set_title(f'Detection: "{ACTIVE}"  score={fp_scores[best_i]:.3f}'); ax1.axis('off')
d0 = fp_depth.astype(float); d0[d0==0] = np.nan
ax2.imshow(d0, cmap='plasma'); ax2.set_title('Depth at detection pos'); ax2.axis('off')
plt.tight_layout(); plt.show()

# ── DeliGrasp: fire in background so it overlaps GraspGenX + arbiter + moves ─
# It only needs the detection image and object name — no angle, no arm state.
# By the time we need force params (right before grip), it's already done.
import threading as _threading
_NUM = r'([0-9]+(?:\.[0-9]+)?)'
def _parse_dg(txt):
    import re as _re2
    m = _re2.search(r'\[start of description\](.*?)\[end of description\]', txt, _re2.DOTALL | _re2.IGNORECASE)
    b = m.group(1) if m else txt
    def g(pat): h = _re2.search(pat, b, _re2.IGNORECASE); return float(h.group(1)) if h else None
    mass = g(rf'mass of {_NUM} grams'); k = g(rf'spring constant of {_NUM} Newtons')
    mu = g(rf'friction coefficient of {_NUM}')
    fi = min(16., max(0.15, (mass/1000*9.81)/max(mu,1e-6))) if mass and mu else 3.
    fc = float(np.clip((k or 1000.)*0.008, 1.0, 16.))
    fi = min(fi, fc)
    return dict(mass_g=mass, k=k, mu=mu, initial_force=fi, force_cap=fc)
_DG_SYS = (
    'Control a robot gripper. Max 16N min 0.15N, max aperture 105mm min 1mm.\n'
    'Rules: replace [PNUM] with a positive number, {CHOICE:[a,b]} with one choice.\n'
    'Spring constant must be between 20 N/m (very soft/squishy) and 2000 N/m (hard rubber/stiff plastic).\n'
    '[start of description]\n'
    '* This is a new grasp.\n'
    '* In accordance with the user instruction, this grasp should be [GRASP_DESCRIPTION].\n'
    '* This is a complete grasp.\n'
    '* This grasp is for an object with {CHOICE:[high,medium,low]} weight.\n'
    '* The object has an approximate mass of [PNUM] grams\n'
    '* This grasp is for an object with {CHOICE:[high,medium,low]} compliance.\n'
    '* The object has an approximate spring constant of [PNUM] Newtons per meter.\n'
    '* The gripper and object have an approximate friction coefficient of [PNUM]\n'
    '* This grasp should set the goal aperture to [PNUM] mm.\n'
    '* If the gripper slips, this grasp should close an additional [PNUM] mm.\n'
    '* If the gripper slips, this grasp should increase the output force by [PNUM] Newtons.\n'
    '[end of description]')
_fast_prior = gm.get_prior(ACTIVE) if 'gm' in dir() else {'n': 0, 'force_mean': None, 'force_std': None}
_skip_deligrasp = (_fast_prior['n'] >= 5 and
                   _fast_prior.get('force_std') is not None and
                   _fast_prior['force_std'] < 1.0)
_dg_result = {}
_dg_thread = None
if _skip_deligrasp:
    _mf = _fast_prior['force_mean']
    _dg_result['GP'] = {'mass_g': None, 'mu': None, 'k': None,
                        'initial_force': _mf,
                        'force_cap': float(np.clip(_mf * 2.5, _mf * 1.1, 16.))}
    print(f'  [FAST] Memory confident (n={_fast_prior["n"]}, '
          f'std={_fast_prior["force_std"]:.2f}N) — skipping DeliGrasp')
elif 'gc' in dir():
    _, _dg_raw_buf = cv2.imencode('.jpg', cv2.cvtColor(fp_color, cv2.COLOR_RGB2BGR))
    _dg_bytes = _dg_raw_buf.tobytes()   # copy now before fp_color is overwritten by scan 2
    _active_dg = str(ACTIVE)
    def _dg_worker(_bytes=_dg_bytes, _name=_active_dg, _res=_dg_result):
        try:
            _raw = gc.models.generate_content(
                model='gemini-2.5-flash',
                contents=[gtypes.Part.from_bytes(data=_bytes, mime_type='image/jpeg'),
                          f'Pick up the {_name}. Assess material and compliance from the image. '
                          'Respond using the exact template, replacing all [PNUM] fields with numbers.'],
                config=gtypes.GenerateContentConfig(system_instruction=_DG_SYS)).text
            _gp = _parse_dg(_raw)
            try:
                _st = gtypes.Tool(google_search=gtypes.GoogleSearch())
                _sl = gc.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=[gtypes.Part.from_bytes(data=_bytes, mime_type='image/jpeg'),
                              f'Object: "{_name}". Search for real specs. Report ONLY:\n'
                              'MASS_G: <grams>\nFRICTION: <vs rubber 0.25-0.8>\nSTIFFNESS: <N/m 20-2000>'],
                    config=gtypes.GenerateContentConfig(tools=[_st])).text
                import re as _re3
                _sm = _re3.search(rf'MASS_G:\s*{_NUM}', _sl, _re3.I)
                _sf = _re3.search(rf'FRICTION:\s*{_NUM}', _sl, _re3.I)
                _sk = _re3.search(rf'STIFFNESS:\s*{_NUM}', _sl, _re3.I)
                if _sm: _gp['mass_g'] = float(_sm.group(1))
                if _sf: _gp['mu'] = float(np.clip(float(_sf.group(1)), 0.15, 1.0))
                if _sk: _gp['k'] = float(np.clip(float(_sk.group(1)), 20., 2000.))
                if _gp.get('mass_g') and _gp.get('mu'):
                    _gp['initial_force'] = min(16., max(0.15, (_gp['mass_g']/1000.*9.81)/max(_gp['mu'],1e-6)))
                    if _gp.get('k'):
                        _ki = float(_gp['k'])
                        _gp['force_cap'] = float(np.clip(_ki * (0.012 if _ki < 400 else 0.025), 1.0, 16.))
            except Exception as _se2:
                _res['search_warn'] = str(_se2)
            _res['GP'] = _gp
        except Exception as _dge:
            _res['error'] = str(_dge)
    _dg_thread = _threading.Thread(target=_dg_worker, daemon=True)
    _dg_thread.start()
    print('  [DeliGrasp] fired in background — running while GraspGenX + arm move...')

_fx, _fy = fp_ci.k[0], fp_ci.k[4]; _cx, _cy = fp_ci.k[2], fp_ci.k[5]

# ── 3. Rough object XY — used only to position arm above object ───────────────
_ys1, _xs1 = np.where(fp_mask)
_u1, _v1   = int(_xs1.mean()), int(_ys1.mean())
_vd1 = fp_depth[fp_mask].astype(float); _vd1 = _vd1[_vd1 > 0]
_dm1 = float(np.median(_vd1)) / 1000. if len(_vd1) > 0 else 0.
_p_cam1   = np.array([(_u1-_cx)*_dm1/_fx, (_v1-_cy)*_dm1/_fy, _dm1])
_p_world1 = (fp_tcp @ _TCP_TO_CAM @ np.array([*_p_cam1, 1.]))[:3]
print(f'  Rough object position: ({_p_world1[0]:.3f}, {_p_world1[1]:.3f}, {_p_world1[2]:.3f})')

# Build scan 1 PCD from detection position — always, so we can merge later
_hint1 = (fp_tcp @ _TCP_TO_CAM @ np.array([
    (_u1-_cx)*_dm1/_fx, (_v1-_cy)*_dm1/_fy, _dm1, 1.]))[:3]
_, _pcd_raw1 = build_segmented_pcd(fp_mask, fp_depth, fp_ci.k, fp_tcp, _TCP_TO_CAM)
_, pts_cln1  = denoise_pcd(_pcd_raw1, hint=_hint1)
print(f'  Scan 1 (detection pos): {len(pts_cln1)} pts')

# ── 4. Move directly above object and scan from top-down ─────────────────────
tcp2 = fp_tcp.copy()
tcp2[0, 3] = _p_world1[0]; tcp2[1, 3] = _p_world1[1]
tcp2[2, 3] = _p_world1[2] + APPROACH_H + GRIPPER_LEN
node.unteach(); node.move(tcp2, spd=0.08); node.spin(4)

# ── Verify TOP-DOWN before scanning (CHECK ONLY — no auto-move). Auto-centring
# diverges while _TCP_TO_CAM is imperfect (each move re-projects the object), so we
# just report how centred the view is. Recalibrate to re-enable closed-loop centring.
node.spin(4)
color2 = node.color.copy(); depth2 = node.depth.copy()
_, _, mask2 = sam3_query(color2, ACTIVE)
if mask2 is None or not mask2.any():
    mask2 = fp_mask
    print('  SAM3 scan 2: no mask — reusing detection mask')
else:
    _ys2, _xs2 = np.where(mask2)
    _h2, _w2 = mask2.shape
    _off_px = float(np.hypot(_xs2.mean() - _w2/2., _ys2.mean() - _h2/2.))
    print(f'  top-down view: object {_off_px:.0f}px from image centre'
          + ('  ✓' if _off_px < 70 else '  (off-centre — _TCP_TO_CAM needs recal)'))

_ys2, _xs2 = np.where(mask2)
_u2, _v2   = int(_xs2.mean()), int(_ys2.mean())
_vd2 = depth2[mask2].astype(float); _vd2 = _vd2[_vd2 > 0]
_dm2 = float(np.median(_vd2)) / 1000. if len(_vd2) > 0 else 0.
_hint2 = (tcp2 @ _TCP_TO_CAM @ np.array([
    (_u2-_cx)*_dm2/_fx, (_v2-_cy)*_dm2/_fy, _dm2, 1.]))[:3]
_, pcd_raw2 = build_segmented_pcd(mask2, depth2, fp_ci.k, tcp2, _TCP_TO_CAM)
_, pts_cln2 = denoise_pcd(pcd_raw2, hint=_hint2)
vq2 = check_view_quality(pts_cln2, mask2, depth2)
print(f'  Scan (top-down): {vq2["n_pts"]} pts  fill={vq2["fill"]:.0%}  {"OK" if vq2["ok"] else "BLOCKED"}')

# SAM3 and GraspGenX COEXIST on the 8 GB card (measured: SAM3 ~3.9 GB + GraspGenX
# ~0.8 GB, peak ~4.8 GB during inference — ~3 GB headroom). Both stay warm all
# session; no VRAM swap, so SAM3 stays available for the centering check + live
# recalibration, and the GraspGenX server is reused (no per-grasp reload).

# ── Merge scan 1 + scan 2 for better coverage on irregular objects ────────
# Top-down sees the footprint; detection-side sees profile/height.
# Merged PCA is more accurate for fruits, asymmetric shapes.
if vq2['ok'] and len(pts_cln1) >= 10 and len(pts_cln2) >= 10:
    # Two scans, two arm poses: the detection scan, then the arm moved CLOSER for a
    # top-down scan. Each is already transformed to world by its own TCP pose, but
    # residual hand-eye calibration drift still offsets the SAME object between them.
    # Align scan 2 onto scan 1 by XY centroid (scan 1 = detection scan = the trusted
    # localisation) so the object OVERLAYS into one dense cloud instead of ghosting,
    # then merge. This accounts for the move-closer difference in the merged result.
    _c1 = pts_cln1.mean(axis=0)
    _c2 = pts_cln2.mean(axis=0)
    _reg_err = float(np.linalg.norm((_c1 - _c2)[:2]))
    _shift = np.array([_c1[0] - _c2[0], _c1[1] - _c2[1], 0.])   # align XY, keep Z
    pts_cln2a = pts_cln2 + _shift
    _pcd_both = o3d.geometry.PointCloud()
    _pcd_both.points = o3d.utility.Vector3dVector(np.vstack([pts_cln1, pts_cln2a]))
    _, pts_use = denoise_pcd(_pcd_both, hint=_c1)
    mask_use = mask2
    src_label = 'merged'
    print(f'  Merged (aligned {_reg_err*1000:.1f}mm pose/calib drift): '
          f'{len(pts_cln1)} + {len(pts_cln2)} → {len(pts_use)} pts, '
          f'anchored to detection scan')
elif vq2['ok']:
    pts_use = pts_cln2; mask_use = mask2; src_label = 'top-down'
else:
    print('  Top-down blocked — using detection-position scan only')
    pts_use = pts_cln1 if len(pts_cln1) >= len(pts_cln2) else pts_cln2
    mask_use = fp_mask; src_label = 'side-only'

# Drop disconnected stray clusters (IR-stereo depth artifacts, table specks) that
# survive statistical denoise — keep only the largest connected blob (the object),
# so the centroid and PCA see the object alone, not a phantom clump beside it.
if len(pts_use) > 40:
    _pcf = o3d.geometry.PointCloud()
    _pcf.points = o3d.utility.Vector3dVector(pts_use)
    _lbl = np.asarray(_pcf.cluster_dbscan(eps=0.012, min_points=10))
    if _lbl.max() >= 0:
        _big  = np.bincount(_lbl[_lbl >= 0]).argmax()
        _keep = _lbl == _big
        if _keep.sum() >= 0.5 * len(pts_use):     # safety: never nuke the object
            _drop = int((~_keep).sum())
            if _drop:
                pts_use = pts_use[_keep]
                print(f'  Cluster filter: dropped {_drop} stray pts '
                      f'({_lbl.max()+1} clusters → kept largest)')

if len(pts_use) < 3:
    raise RuntimeError(f'Too few points after denoise: {len(pts_use)}')

# ── Show: scan 1 | scan 2 | merged comparison ───────────────────────────
fig = plt.figure(figsize=(18, 5))
_ax1 = fig.add_subplot(131, projection='3d')
if len(pts_cln1) > 0:
    _ax1.scatter(pts_cln1[:,0], pts_cln1[:,1], pts_cln1[:,2],
                 c=pts_cln1[:,2], cmap='plasma', s=4, alpha=.6)
_ax1.set_title(f'Scan 1 (detection)  {len(pts_cln1)} pts')
_ax1.set_xlabel('X'); _ax1.set_ylabel('Y'); _ax1.set_zlabel('Z')
_ax2 = fig.add_subplot(132, projection='3d')
if len(pts_cln2) > 0:
    _ax2.scatter(pts_cln2[:,0], pts_cln2[:,1], pts_cln2[:,2],
                 c=pts_cln2[:,2], cmap='viridis', s=4, alpha=.6)
_ax2.set_title(f'Scan 2 (top-down)  {len(pts_cln2)} pts')
_ax2.set_xlabel('X'); _ax2.set_ylabel('Y'); _ax2.set_zlabel('Z')
_ax3 = fig.add_subplot(133)
_ax3.scatter(pts_use[:,0], pts_use[:,1], c=pts_use[:,2], cmap='viridis', s=5, alpha=.7)
_ax3.set_title(f'Used [{src_label}]  {len(pts_use)} pts')
_ax3.set_xlabel('X m'); _ax3.set_ylabel('Y m'); _ax3.set_aspect('equal')
plt.suptitle('Point cloud scans'); plt.tight_layout(); plt.show()


fp_color = color2; fp_depth = depth2; fp_tcp = tcp2; fp_mask = mask2

# ── 5. PCA + mask-based angle (stripe-free) ───────────────────────────────────
pts_top  = top_layer(pts_use)  # PCA/angle only — do NOT augment from pts_top; augment_cloud_for_graspgenx uses raw pts_use for correct height
pca      = analyse_pcd(pts_top)
cen      = pca['centroid']
ext      = pca['extent_m']
obj_w_mm = ext[1] * 1000.
ang, strategy, reason = smart_grasp_angle(
    pca, object_name=ACTIVE, image_rgb=fp_color,
    gemini_client=gc, mask=mask_use)
print(f'  PCA: major={ext[0]*1e3:.1f}mm  minor={ext[1]*1e3:.1f}mm  angle={ang:.1f}deg  [{strategy}]')
print(f'  reason: {reason}')
_ang_pca = float(ang)   # raw PCA angle — baseline for the angle feedback loop
# ── GraspGenX: fire in background thread IMMEDIATELY — runs concurrently with ──
# PCA viz + depth centroid + safety check below (~350ms of free GPU time). The
# server is warm (preloaded at startup, coexists with SAM3 in VRAM), so this is
# just a ~0.5s ZMQ inference overlapped with the main-thread PCA work.
_GMETHOD_raw = GRASP_METHOD if 'GRASP_METHOD' in dir() else 'pca'
_GMETHOD     = _GMETHOD_raw
_ang_ggx     = None
_ggx_clusters = []
_ggx_result  = {}   # thread writes here; main thread reads after join

def _ggx_worker(_pts=pts_use, _mask=mask_use, _tcp=fp_tcp, _k=fp_ci.k,
                _tz=TABLE_Z, _cam_h=CAMERA_MOUNT_Z_OFFSET_M, _strat=strategy,
                _res=_ggx_result):
    try:
        sys.path.insert(0, os.path.expanduser('~/magpie_control/scripts/grasp_detectors'))
        from grasp_detectors import socket_detector as _sdc
        _det = {'graspgen': _sdc.graspgen,
                'graspgenx': _sdc.graspgenx}[_GMETHOD_raw]()
        if not _det.available():
            print(f'  [{_GMETHOD_raw}] server not up — starting automatically...')
            _ggx_sh = os.path.expanduser('~/magpie_control/scripts/run_graspgenx_server.sh')
            if 'PROCS' in dir():
                PROCS['graspgenx'] = subprocess.Popen(
                    ['bash', _ggx_sh],
                    stdout=open('/tmp/log_graspgenx.txt', 'w'), stderr=subprocess.STDOUT)
            for _wi in range(60):
                time.sleep(1.0)
                if _det.available(): print(f'  [{_GMETHOD_raw}] ready in {_wi+1}s'); break
            else:
                _res['error'] = 'server failed to start'; return
        _pts_ggx = augment_cloud_for_graspgenx(_pts, _tz, extra_height_m=_cam_h)
        _gr = _det.detect(_pts_ggx, mask=_mask, tcp=_tcp, intrinsics=_k)
        if not _gr.best:
            _res['no_grasp'] = True; return
        _obj_ctr = np.median(_pts[:, :2], axis=0)
        _obj_rad = float(np.percentile(
            np.hypot(_pts[:,0]-_obj_ctr[0], _pts[:,1]-_obj_ctr[1]), 90)) + 0.005
        _valid = [g for g in _gr.grasps
                  if np.hypot(g.pose[0,3]-_obj_ctr[0], g.pose[1,3]-_obj_ctr[1]) <= _obj_rad]
        _pool = _valid if _valid else _gr.grasps
        if _strat == 'symmetric':
            _ctr_r = _obj_rad * 0.35
            _ctr_ok = [g for g in _pool
                       if np.hypot(g.pose[0,3]-_obj_ctr[0],
                                   g.pose[1,3]-_obj_ctr[1]) < _ctr_r]
            _pool = _ctr_ok if _ctr_ok else _pool
            print(f'  [ggx sym] {len(_ctr_ok)} centre grasps (r<{_obj_rad*0.35*1000:.0f}mm)')
        _clusters = []
        _used = [False] * len(_pool)
        _near = None
        for _gi, _g in enumerate(sorted(_pool, key=lambda g: -g.score)):
            if _used[_gi]: continue
            _a = float(_g.angle_deg()) % 180.
            if _near is None: _near = _g
            for _gj, _go in enumerate(_pool):
                if _used[_gj]: continue
                if min(abs((float(_go.angle_deg())%180.)-_a), 180.-abs((float(_go.angle_deg())%180.)-_a)) < 15.:
                    _used[_gj] = True
            _clusters.append(min(_a, 90.))
            if len(_clusters) >= 3: break
        _res.update(dict(
            gr=_gr, valid=_valid, obj_ctr=_obj_ctr, obj_rad=_obj_rad,
            clusters=_clusters, ang_ggx=_clusters[0] if _clusters else None,
            near=_near if _near else _gr.best, pool=_pool,
        ))
    except Exception as _e:
        _res['error'] = str(_e)

_ggx_thread = None
if _GMETHOD != 'pca':
    _ggx_thread = _threading.Thread(target=_ggx_worker, daemon=True)
    _ggx_thread.start()
    print(f'  [{_GMETHOD}] inference started in background — PCA viz runs concurrently')


# ── Orientation feedback loop (VLA) — avoid orientations that slipped before ──
# Symmetric-ish footprint (minor/major > 0.75) means rotating 90deg is a valid
# alternative grip. If this angle has a losing track record, switch it.
_sym = (ext[1] / max(ext[0], 1e-6)) > 0.75
_adv = gm.get_angle_advice(ACTIVE, ang, symmetric=_sym) if 'gm' in dir() else None
if _adv and _adv['overridden']:
    print(f'  [orientation memory] {_adv["reason"]}')
    print(f'    -> overriding angle {ang:.1f}deg -> {_adv["recommend_angle"]:.1f}deg')
    ang = float(_adv['recommend_angle'])
elif _adv and _adv['n_at_proposed'] > 0:
    print(f'  [orientation memory] {_adv["reason"]} (keeping angle)')

# ── Visual grasp verification — draw the grasp axis on the camera image (projected
#    from world coords so it lines up with the object) and let Gemini confirm or
#    correct it. Catches degenerate cases (e.g. diagonal grasp on a square cube).
# ── Gripper-angle feedback loop — apply the learned PCA->good-angle correction.
#    Pose-invariant delta learned from past HELD grasps of this object. Once
#    confident, trust it and skip the per-grasp Gemini visual check (like the
#    DeliGrasp skip for force). The visual check still runs while unconfident,
#    and every outcome is recorded so the correction keeps improving.
_skip_visual = False
_acorr = gm.get_angle_correction(ACTIVE) if 'gm' in dir() else None
if _acorr:
    print(f'  [angle memory] learned correction {_acorr["delta"]:+.1f}deg from '
          f'{_acorr["n"]} held grasps (q={_acorr["mean_quality"]:.2f}, '
          f'spread={_acorr["spread_deg"]:.1f}deg)')
    ang = float((_ang_pca + _acorr['delta']) % 90.)
    if _acorr['confident']:
        _skip_visual = True
        print(f'    confident -> applying learned angle {ang:.1f}deg, skipping visual check')

_grasp_vis = None; _grasp_verdict = 'none'; _grasp_check_reason = ''
# Gemini visual check MOVED below: it now runs AFTER GraspGenX so it arbitrates the
# FINAL PCA-vs-GraspGenX candidates right before descent (no more verify-then-override).

# ── Show: PCA ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 5))
axp1 = fig.add_subplot(121, projection='3d')
axp1.scatter(pts_use[:,0], pts_use[:,1], pts_use[:,2], c=pts_use[:,2], cmap='plasma', s=4, alpha=.3)
ptc  = pts_use - cen
ev, evec = np.linalg.eigh(np.cov(ptc.T))
idx  = np.argsort(ev)[::-1]; evec = evec[:,idx]; sc_e = np.sqrt(np.abs(ev[idx]))
for i, (lbl, col) in enumerate(zip(['Major','Minor','Normal'],['red','green','blue'])):
    axp1.quiver(*cen, *(evec[:,i]*sc_e[i]), color=col, lw=2.5, label=lbl)
axp1.scatter(*cen, color='yellow', s=120, zorder=5)
axp1.set_xlabel('X'); axp1.set_ylabel('Y'); axp1.set_zlabel('Z')
axp1.set_title(f'PCA  angle={ang:.1f}deg'); axp1.legend(fontsize=8)
axp2 = fig.add_subplot(122)
axp2.scatter(pts_use[:,0], pts_use[:,1], c=pts_use[:,2], cmap='plasma', s=5, alpha=.5)
t_a = np.radians(ang); L_a = sc_e[0]
axp2.annotate('', xy=(cen[0]+L_a*np.cos(t_a), cen[1]+L_a*np.sin(t_a)),
              xytext=(cen[0]-L_a*np.cos(t_a), cen[1]-L_a*np.sin(t_a)),
              arrowprops=dict(arrowstyle='<->', color='red', lw=2))
axp2.annotate('', xy=(cen[0]+L_a*np.cos(t_a+np.pi/2), cen[1]+L_a*np.sin(t_a+np.pi/2)),
              xytext=cen[:2], arrowprops=dict(arrowstyle='->', color='lime', lw=2))
axp2.set_aspect('equal'); axp2.set_xlabel('X m'); axp2.set_ylabel('Y m')
axp2.set_title('Top-down: major axis + gripper')
import io as _io
_pca_buf = _io.BytesIO()
fig.savefig(_pca_buf, format='jpeg', dpi=80, bbox_inches='tight')
_pca_plot_bytes = _pca_buf.getvalue()
plt.tight_layout(); plt.show()

# ── 6. Depth centroid + TABLE_Z ───────────────────────────────────────────────
ys, xs = np.where(fp_mask)
u, v   = int(xs.mean()), int(ys.mean())
valid  = fp_depth[fp_mask].astype(float); valid = valid[valid > 0]
dm     = float(np.median(valid)) / 1000.
p_cam  = np.array([(u-_cx)*dm/_fx, (v-_cy)*dm/_fy, dm])
T      = fp_tcp @ _TCP_TO_CAM
p_obj  = (T @ np.array([*p_cam, 1.]))[:3]
# Grasp XY from the MERGED cloud centroid (anchored to the detection scan), NOT the
# top-down depth centroid — top-down localisation drifts under calibration error and
# was missing the object. Keep Z from the depth centroid for the grasp-depth calc.
_p_obj_td = p_obj.copy()
p_obj[0], p_obj[1] = float(cen[0]), float(cen[1])
if np.linalg.norm(_p_obj_td[:2] - p_obj[:2]) > 0.01:
    print(f'  grasp XY -> merged-cloud centroid ({p_obj[0]:.3f},{p_obj[1]:.3f}) '
          f'[top-down depth said ({_p_obj_td[0]:.3f},{_p_obj_td[1]:.3f})]')
print(f'  p_obj=({p_obj[0]:.3f},{p_obj[1]:.3f},{p_obj[2]:.3f})')

# Always use TABLE_Z measured at notebook start (before object placed).
# Re-estimating from live scene is unreliable with object in view.
_tz = TABLE_Z
if _tz is None:
    print('  [!] TABLE_Z not set — using HARD_FLOOR_Z. Run the Z-scan cell first.')
    _tz = HARD_FLOOR_Z
obj_height_est = max(p_obj[2] - _tz, 0.010)
grasp_off      = float(np.clip(obj_height_est / 2., 0.010, 0.040))
print(f'  TABLE_Z={_tz*1000:.1f}mm  obj_h={obj_height_est*1000:.1f}mm  grasp_off={grasp_off*1000:.1f}mm')

# ── 7. Safety check ───────────────────────────────────────────────────────────
afz = p_obj[2] + APPROACH_H
gfz = p_obj[2] - grasp_off
# Raise gfz to compensate for fingertip arc-drop during close.
# Fingers swing down ~21mm fully open→closed; compensate based on expected grip aperture.
_grip_ap_mm  = float(np.clip(pca['extent_m'][1] * 1000. + 5., 5., 90.))
_arc_comp_m  = fingertip_drop(105., _grip_ap_mm)
gfz         += _arc_comp_m
print(f'  arc_comp=+{_arc_comp_m*1000:.1f}mm (close_ap≈{_grip_ap_mm:.0f}mm)  new gfz={gfz*1000:.1f}mm')
assert afz >= HARD_FLOOR_Z, f'Approach fingertip {afz:.3f}m below floor'
assert gfz >= HARD_FLOOR_Z, f'Grasp fingertip {gfz:.3f}m below floor'
print(f'Safety OK  approach_fz={afz:.3f}m  grasp_fz={gfz:.3f}m')

# ── Join GraspGenX thread + visualise results ────────────────────────────────
if _ggx_thread is not None:
    _ggx_thread.join(timeout=30.)
    if _ggx_thread.is_alive():
        print(f'  [{_GMETHOD}] timed out — falling back to PCA'); _GMETHOD = 'pca'
    elif _ggx_result.get('error'):
        print(f'  [{_GMETHOD}] error: {_ggx_result["error"]} — falling back to PCA'); _GMETHOD = 'pca'
    elif _ggx_result.get('no_grasp'):
        print(f'  [{_GMETHOD}] no grasp — falling back to PCA'); _GMETHOD = 'pca'
    else:
        _gr      = _ggx_result['gr']
        _valid   = _ggx_result['valid']
        _obj_ctr = _ggx_result['obj_ctr']
        _obj_rad = _ggx_result['obj_rad']
        _ggx_clusters = _ggx_result['clusters']
        _ang_ggx = _ggx_result['ang_ggx']
        _near    = _ggx_result['near']
        _cx, _cy = float(_obj_ctr[0]), float(_obj_ctr[1])
        p_obj = p_obj.copy(); p_obj[0], p_obj[1] = _cx, _cy
        print(f'  [{_GMETHOD}] {_gr.n} grasps, {len(_valid)} on-object | '
              f'{len(_ggx_clusters)} clusters: {[f"{a:.0f}°" for a in _ggx_clusters]} '
              f'centre=({_cx:.3f},{_cy:.3f}) lat={1000*_gr.latency_s:.0f}ms')
        if not _valid:
            print(f'  [{_GMETHOD}] !! ALL grasps off-object — using centre + clusters')
        # Visualise (must be on main thread)
        _topk = _gr.grasps[:min(30, _gr.n)]
        _sc   = np.array([g.score for g in _topk])
        _pos  = np.array([[g.pose[0,3], g.pose[1,3]] for g in _topk])
        _cls  = np.array([[g.pose[0,0], g.pose[1,0]] for g in _topk])
        _norm_sc = plt.Normalize(_sc.min(), _sc.max()); _cmap_sc = plt.cm.RdYlGn
        _hm = 0.018
        fig, (ax_g, ax_s) = plt.subplots(1, 2, figsize=(14, 5))
        ax_g.scatter(pts_use[:,0], pts_use[:,1], c=pts_use[:,2], cmap='Blues', s=4, alpha=0.35)
        for _i, (p2, c2, s2) in enumerate(zip(_pos, _cls, _sc)):
            ax_g.annotate('', xy=(p2[0]+c2[0]*_hm, p2[1]+c2[1]*_hm),
                          xytext=(p2[0]-c2[0]*_hm, p2[1]-c2[1]*_hm),
                          arrowprops=dict(arrowstyle='<->', color=_cmap_sc(_norm_sc(s2)),
                                         lw=3 if _i==0 else 0.8))
        ax_g.plot(_cx, _cy, '+', color='magenta', ms=22, mew=3, label='grasp centre')
        ax_g.add_patch(plt.Circle((_cx,_cy),_obj_rad,fill=False,color='magenta',ls='--',lw=1))
        ax_g.plot(_near.pose[0,3], _near.pose[1,3], '*', color='gold', ms=15,
                  label=f'angle src ({_near.score:.2f})')
        ax_g.set_xlabel('X m'); ax_g.set_ylabel('Y m'); ax_g.set_aspect('equal')
        ax_g.set_title(f'GraspGenX top-{len(_topk)} (best ★)'); ax_g.legend()
        _all_sc = sorted([g.score for g in _gr.grasps], reverse=True)[:30]
        ax_s.bar(range(len(_all_sc)), _all_sc,
                 color=[_cmap_sc(_norm_sc(s)) for s in _all_sc])
        ax_s.axhline(_gr.best.score, color='gold', ls='--', lw=2,
                     label=f'best={_gr.best.score:.3f}')
        ax_s.set_xlabel('rank'); ax_s.set_ylabel('score'); ax_s.set_title('Score dist'); ax_s.legend()
        plt.suptitle(f'GraspGenX — {_gr.n} grasps | {_ang_ggx:.1f}° | {1000*_gr.latency_s:.0f}ms')
        plt.tight_layout(); plt.show()

# ── Gemini MULTI-CANDIDATE ARBITER ──────────────────────────────────────────────
# Build a pool of up to 4 clamped angle candidates, show them all to Gemini as
# labelled lines on the real image, and let it rank + fine-tune the best one.
# Candidates: PCA-short, PCA-long (perpendicular), then up to 2 GraspGenX clusters.
# All angles are wrist-clamped (0–90°) so Gemini sees exactly what will execute.
_ang_arb_choice = 'pca-only'
if 'gc' in dir() and not _skip_visual:
    try:
        from pointcloud_utils import verify_grasp_angle_visual
        _half = float(max(0.03, ext[0] * 0.6))
        _ang_pca_arb = min(float(ang) % 180., 90.)   # wrist-clamped PCA angle
        # Build candidate pool: PCA + GraspGenX clusters (all clamped)
        _arb_candidates = [('PCA', _ang_pca_arb)]
        if '_ggx_clusters' in dir():
            for _ci, _ca in enumerate(_ggx_clusters[:3]):
                # only add if meaningfully different from existing candidates
                if all(abs((_ca - _ea + 90.) % 180. - 90.) > 8.
                       for _, _ea in _arb_candidates):
                    _arb_candidates.append((f'GGX-{_ci+1}', _ca))
        print(f'  [arbiter] {len(_arb_candidates)} candidates: '
              f'{[(l, f"{a:.0f}°") for l, a in _arb_candidates]}')
        if len(_arb_candidates) >= 2:
            _ang_arb, _chosen, _grasp_check_reason, _gv = rank_grasp_angles_visual(
                fp_color, cen, _arb_candidates, fp_tcp, _TCP_TO_CAM, fp_ci.k,
                object_name=ACTIVE, gemini_client=gc, half_len_m=_half, mask=mask_use)
            _ang_arb_choice = _chosen; _grasp_verdict = f'CHOSE {_chosen}'
            print(f'  [arbiter] Gemini chose {_chosen} = {_ang_arb:.1f}deg: {_grasp_check_reason}')
            ang = min(float(_ang_arb) % 180., 90.)
            _grasp_title = f'Gemini arbiter ({len(_arb_candidates)} opts) -> {_chosen}'
        else:
            # Only one candidate — just verify/nudge it
            _ang_new, _grasp_verdict, _grasp_check_reason, _gv = verify_grasp_angle_visual(
                fp_color, cen, ang, fp_tcp, _TCP_TO_CAM, fp_ci.k,
                object_name=ACTIVE, gemini_client=gc, half_len_m=_half, mask=mask_use)
            _ang_arb_choice = 'pca'
            if _grasp_verdict.startswith('ROTATE') and abs((_ang_new - ang + 90.) % 180. - 90.) > 3.:
                print(f'  [grasp check] corrected {ang:.1f}deg -> {_ang_new:.1f}deg')
                ang = min(float(_ang_new) % 180., 90.)
            print(f'  [grasp check] {_grasp_verdict}: {_grasp_check_reason}')
            _grasp_title = f'Grasp check [{_grasp_verdict}] — green=close axis, orange=fingers'
        _grasp_vis = cv2.cvtColor(_gv, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(5, 4)); plt.imshow(_grasp_vis)
        plt.title(_grasp_title); plt.axis('off'); plt.show()
    except Exception as _ve:
        print(f'  [arbiter skipped: {_ve}]')

# ── 8. Poses ──────────────────────────────────────────────────────────────────
ap = tcp2.copy()
ap[0, 3] = p_obj[0]; ap[1, 3] = p_obj[1]
ap[2, 3] = p_obj[2] + APPROACH_H + GRIPPER_LEN
ang = min(float(ang) % 180., 90.)   # wrist safety: clamp to [0,90]
R_grasp    = grasp_rotation_matrix(ang)
ap_rot     = ap.copy(); ap_rot[:3, :3] = R_grasp
grasp_pose = ap_rot.copy(); grasp_pose[2, 3] = gfz + GRIPPER_LEN

# ── 9. Execute ────────────────────────────────────────────────────────────────
# This section is a general execution primitive — no task-specific policy.
# Force parameters come from GP (c14 / DeliGrasp). The operator or a future
# VLA policy decides whether to re-run with different parameters.

# ── Await DeliGrasp result (fired at detection time, ran in background) ──────
if _dg_thread is not None:
    _dg_thread.join(timeout=20.)
    if _dg_thread.is_alive():
        print('  [DeliGrasp] timed out after 20s — using memory prior only')
    elif _dg_result.get('search_warn'):
        print(f'  [search grounding failed: {_dg_result["search_warn"]}]')
GP = _dg_result.get('GP')
if GP and not _skip_deligrasp:
    print(f'  DeliGrasp: mass={GP.get("mass_g")}g  k={GP.get("k")}N/m  '
          f'mu={GP.get("mu")}  F0={GP.get("initial_force",0):.2f}N  '
          f'cap={GP.get("force_cap",16):.2f}N')

# Retrieve force prior from memory (DINO RAG → string → none)
_prior = gm.get_prior(ACTIVE, _det_crop) if 'gm' in dir() else {'force_mean': None, 'n': 0, 'source': 'none'}
if _prior['force_mean'] is not None:
    print(f'  Memory prior: {_prior["force_mean"]:.1f}N ± {_prior["force_std"]:.2f}N '
          f'(n={_prior["n"]} grasps, source={_prior["source"]})')

if 'GP' in dir() and GP and GP.get('mass_g') and GP.get('mu'):
    F_min       = (GP['mass_g'] / 1000. * 9.81) / (2. * max(GP['mu'], 1e-6))
    force_cap   = float(GP.get('force_cap', 16.))
    if GP.get('k'):
        _k = float(GP['k'])
        # Soft (k<400): 12mm crush limit. Stiff (k≥400): 25mm — structure handles it.
        _deform_m = 0.012 if _k < 400 else 0.025
        force_cap = float(np.clip(_k * _deform_m, 1.0, 16.))
    cf_gp       = float(np.clip(max(GP['initial_force'], F_min * 1.5), 0.5, force_cap))
    # Blend: weight memory heavier as n grows (trust prior more after 5+ grasps)
    if _prior['force_mean'] is not None and _prior['n'] >= 2:
        _w = min(0.9, _prior['n'] / 4.)   # 90% memory weight at n=4; GP is sanity-check only
        # Memory can raise force_cap — after ≥3 real grasps it's more reliable than k formula
        if _prior['n'] >= 3:
            force_cap = max(force_cap, _prior['force_mean'] * 1.5)
        cf = float(np.clip(_w * _prior['force_mean'] + (1 - _w) * cf_gp, 0.5, force_cap))
        print(f'  Blended force: {cf:.1f}N (memory {_w:.0%} + GP {1-_w:.0%})')
    else:
        cf = cf_gp
    slip_thresh = float(np.clip(F_min * 0.9, 0.15, force_cap * 0.9))
else:
    F_min = 0.
    if _skip_deligrasp and _prior.get('force_mean'):
        # Fast path: GP was intentionally bypassed — use memory force
        cf         = float(_prior['force_mean'])
        force_cap  = float(np.clip(cf * 2.5, cf * 1.1, 16.))
        slip_thresh = float(np.clip(cf * 0.5, 0.15, force_cap * 0.9))
    elif _prior.get('force_mean'):
        # No GP and no skip flag — best we have is the memory prior
        print('  [!] GP missing — run c14 first. Using memory prior.')
        cf         = float(_prior['force_mean'])
        force_cap  = 8.   # no GP, no memory — conservative cap for unknown object
        slip_thresh = float(np.clip(cf * 0.5, 0.15, force_cap * 0.9))
    else:
        print('  [!!!] No GP and no memory — fallback cf=8N.')
        cf = 4.; force_cap = 8.; slip_thresh = 0.5

print(f'\nGrasp params: force={cf:.1f}N  cap={force_cap:.1f}N  thresh={slip_thresh:.2f}N')

# ── VLA episode recording (LeRobot, 10Hz, gated on reward) ──────────────────
if 'vla_rec' not in dir():
    try:
        from vla_recorder import VLARecorder
        vla_rec = VLARecorder(node, fps=10,
                              rerun=(VLA_RERUN if 'VLA_RERUN' in dir() else None))
    except Exception as _vre:
        print(f'  [VLA] recorder unavailable: {_vre}')
        class _NullRec:
            def __getattr__(self, _): return lambda *a, **k: None
        vla_rec = _NullRec()
vla_rec.start_episode(f'pick up the {ACTIVE}')

# ── Approach ──────────────────────────────────────────────────────────────────
vla_rec.set_action(ap, grip=0.)
node.open_g(); time.sleep(0.5)
print('Approach...')
node.move(ap,       spd=0.08); time.sleep(0.4); node.spin(4)
vla_rec.set_action(ap_rot, grip=0.)
node.move(ap_rot,   spd=0.05); time.sleep(0.4); node.spin(4)

# ── Pre-descent centering check ──────────────────────────────────────────────
# Re-detect the object after wrist rotation. If its centroid drifted off image
# centre, re-localise in world frame from THIS view and CORRECT grasp_pose's XY
# before descending — previously this only printed a warning and still
# descended on the old (possibly off-centre) XY, so the gripper closed lopsided.
# _centering_flag drives the reward gate below: 'ok' (no drift, or fixed by a
# simple re-localise, or fixed by live RECALIBRATION) vs 'unreliable' (nothing
# worked) — only 'unreliable' forces this episode to fail the reward gate.
#
# Escalation ladder when the object is off-centre after wrist rotation:
#   1. Re-localise from this view and snap grasp XY to it (cheap, no arm move).
#   2. If that jump is too large to trust (>60mm), the EXTRINSIC itself is
#      probably off, not just this one detection — call auto_calibrate() live.
#      It re-derives the camera clocking angle from a 30mm jog using the SAME
#      object already on the table (no clearing needed), updates _TCP_TO_CAM
#      for this AND future grasps this session, and persists it to disk.
#   3. Re-localise again with the fixed extrinsic and proceed normally.
#   4. Only if recalibration itself fails/is untrustworthy do we give up and
#      exclude this episode from the trainable set.
_centering_flag = 'ok'
try:
    _pre_col = node.color.copy()
    _pre_depth = node.depth.copy()
    # SAM3 stays alive (coexists with GraspGenX), so re-DETECT the object in the
    # current top-down view and check it's actually centred in frame — this SEES the
    # physical object, unlike FK projection which only re-derives stored maths.
    _, _, _pre_mask = sam3_query(_pre_col, ACTIVE)
    if _pre_mask is not None and _pre_mask.any():
        _ys_p, _xs_p = np.where(_pre_mask)
        _cx_im, _cy_im = node.caminfo.k[2], node.caminfo.k[5]
        _dx_px = float(_xs_p.mean() - _cx_im)
        _dy_px = float(_ys_p.mean() - _cy_im)
        _off_px = float(np.hypot(_dx_px, _dy_px))
        _ctr_ok = _off_px < 80.
        print(f'Centering: ({_dx_px:+.0f}, {_dy_px:+.0f})px from image centre  '
              f'— {"✓ OK" if _ctr_ok else f"⚠ {_off_px:.0f}px off — correcting XY"}')
        if not _ctr_ok:
            _pre_pts, _ = build_segmented_pcd(_pre_mask, _pre_depth, node.caminfo.k,
                                               node.tcp, _TCP_TO_CAM)
            _new_xy = np.median(_pre_pts[:, :2], axis=0) if len(_pre_pts) > 20 else None
            _corr_mm = (float(np.hypot(_new_xy[0] - grasp_pose[0, 3],
                                       _new_xy[1] - grasp_pose[1, 3])) * 1000.
                        if _new_xy is not None else float('inf'))
            if _new_xy is not None and _corr_mm < 60.:
                grasp_pose[0, 3] = ap[0, 3] = ap_rot[0, 3] = float(_new_xy[0])
                grasp_pose[1, 3] = ap[1, 3] = ap_rot[1, 3] = float(_new_xy[1])
                print(f'  -> corrected by {_corr_mm:.0f}mm to '
                      f'({_new_xy[0]:.3f},{_new_xy[1]:.3f})')
            else:
                print(f'  -> simple correction unreliable ({_corr_mm:.0f}mm) — '
                      f'the camera extrinsic is likely off. RECALIBRATING live...')
                try:
                    from auto_calib import auto_calibrate
                    _z_off_now = float(_TCP_TO_CAM[2, 3])
                    _new_T2C, _cinfo = auto_calibrate(node, sam3_query, ACTIVE,
                                                       step=0.03, z_off=_z_off_now)
                    _new_T2C[0, 3] += _CAM_XY_OFFSET_M[0]
                    _new_T2C[1, 3] += _CAM_XY_OFFSET_M[1]
                    print(f'  [recalib] Rz({_cinfo["clocking_deg"]}deg) '
                          f'err={_cinfo["match_err_px"]:.1f}px '
                          f'{"✓ trustworthy" if _cinfo["trustworthy"] else "⚠ still high residual"}')
                    if _cinfo['trustworthy']:
                        _TCP_TO_CAM = _new_T2C   # live update — benefits this AND future grasps
                        import json as _jj4
                        _calib_save2 = _pp.Path('~/magpie_control/data/last_calibration.json').expanduser()
                        _jj4.dump({'clocking_deg': _cinfo['clocking_deg'], 'z_off': _z_off_now,
                                   'cam_xy_offset': list(_CAM_XY_OFFSET_M),
                                   'saved_at': _dt.datetime.now().isoformat()},
                                  open(_calib_save2, 'w'), indent=2)
                        node.spin(6)
                        _col2, _dep2 = node.color.copy(), node.depth.copy()
                        _, _, _mask2 = sam3_query(_col2, ACTIVE)
                        _pts2 = (build_segmented_pcd(_mask2, _dep2, node.caminfo.k,
                                                      node.tcp, _TCP_TO_CAM)[0]
                                 if _mask2 is not None and _mask2.any() else np.zeros((0, 3)))
                        if len(_pts2) > 20:
                            _xy2 = np.median(_pts2[:, :2], axis=0)
                            grasp_pose[0, 3] = ap[0, 3] = ap_rot[0, 3] = float(_xy2[0])
                            grasp_pose[1, 3] = ap[1, 3] = ap_rot[1, 3] = float(_xy2[1])
                            print(f'  -> recalibrated + relocalised to '
                                  f'({_xy2[0]:.3f},{_xy2[1]:.3f}); proceeding normally')
                        else:
                            _centering_flag = 'unreliable'
                            print('  -> lost object after recalib — EXCLUDING from training data')
                    else:
                        _centering_flag = 'unreliable'
                        print('  -> recalibration itself unreliable — EXCLUDING from training data')
                except Exception as _re_err:
                    _centering_flag = 'unreliable'
                    print(f'  -> recalibration failed ({_re_err}) — EXCLUDING from training data')
    else:
        _centering_flag = 'unreliable'
        print('  centering check: object not detected after rotation — EXCLUDING from training data')

except Exception as _ctr_err:
    _centering_flag = 'unreliable'
    print(f'  centering check skipped ({_ctr_err}) — EXCLUDING from training data')

print('Descend...')
vla_rec.set_action(grasp_pose, grip=0.)
node.move(grasp_pose, spd=0.05); time.sleep(0.5); node.spin(4)

# Snapshot at grasp position — what the gripper camera sees right before closing
node.spin(4)
snap_grasp = node.color.copy()

# ── Grasp ─────────────────────────────────────────────────────────────────────
# clear_err → set_force → close_g each attempt.
# +1N per slip is a mechanical characteristic (AX-12 needs authority to seat),
# not a policy decision. 4 attempts max.
def grip_refresh(force_n):
    node.clear_err()
    node.set_force(force_n)
    node.close_g()

# Pre-position gripper just above expected contact — less travel = more AX-12 force
_prepos_mm = max(10., obj_w_mm + 8.)
node.set_force(cf); node.set_pos(_prepos_mm); time.sleep(0.5)

grasp_log = []
vla_rec.set_action(grasp_pose, grip=1.)   # commanding the close
_mem_confident = _prior.get('n', 0) >= 3  # memory has enough data to trust directly
for attempt in range(4):
    grip_refresh(cf)
    time.sleep(2.5)
    for _ in range(8): rclpy.spin_once(node, timeout_sec=0.15)
    s = node.gs
    if s is None: break
    status = 'contact' if s.force >= slip_thresh else 'slip'
    print(f'  attempt {attempt+1}: ap={s.position:.1f}mm  F={s.force:.3f}N  → {status}')
    grasp_log.append({'attempt': attempt+1, 'ap': s.position,
                      'force': s.force, 'cmd_force': cf, 'status': status})
    if status == 'contact': break
    cf = min(force_cap, cf + 1.0)

# ── Lift ──────────────────────────────────────────────────────────────────────
# Slip guard watches depth at object pixel during lift.
# Re-clamps autonomously if object drops >1mm relative to camera.
lift_force = min(force_cap, cf + 0.5)
node.set_force(lift_force)          # tighten grip to lift_force before moving
time.sleep(0.15)
# Scale reclamp step with stiffness: soft objects (low k) get smaller increments
_gp_k = float(GP['k']) if 'GP' in dir() and GP and GP.get('k') else None
_force_step = float(np.clip(_gp_k / 2000., 0.15, 1.0)) if _gp_k else 0.5
node.slip_guard_enable(lift_force, slip_thresh, obj_u=int(u), obj_v=int(v), force_step=_force_step)
print('Lifting...')
vla_rec.set_action(ap_rot, grip=1.)       # holding while lifting
node.move(ap_rot, spd=0.05); time.sleep(0.5); node.spin(8)
s_lift = node.gs
node.slip_guard_disable()
for _ in range(5): rclpy.spin_once(node, timeout_sec=0.1)
snap_held = node.color.copy() if node.color is not None else None

# ── Parse slip guard corrections (DAgger-style training labels) ──────────
import re as _re
_sg_reclamps = [e for e in getattr(node, '_sg_events', []) if 'RECLAMP' in e]
sg_fires = len(_sg_reclamps)
true_force_label = lift_force  # default: no correction needed
if sg_fires:
    _m = _re.search(r'force now ([\d.]+)N', _sg_reclamps[-1])
    if _m:
        true_force_label = float(_m.group(1))
_init_pred = GP.get('initial_force', cf) if 'GP' in dir() and GP else cf
force_label_error = true_force_label - _init_pred
if sg_fires:
    print(f'  Slip guard fired {sg_fires}x — true force label: {true_force_label:.1f}N '
          f'(predicted {_init_pred:.1f}N, error {force_label_error:+.1f}N)')

force_ok    = s_lift is not None and s_lift.force >= slip_thresh
aperture_ok = s_lift is not None and s_lift.position >= obj_w_mm * 0.4
held        = force_ok and aperture_ok
print(f'Post-lift: F={s_lift.force:.3f}N  ap={s_lift.position:.1f}mm  → {"HOLDING" if held else "DROPPED"}')

# ── Result dict ───────────────────────────────────────────────────────────────
# Captures full state for VLA training data collection.
# Policy decisions (retry? different angle? more force?) are made outside this cell.
result = {
    'object':          ACTIVE,
    'held':            held,
    'centering_flag':  _centering_flag if '_centering_flag' in dir() else 'ok',
    'grasp_log':       grasp_log,
    'final_force':     s_lift.force    if s_lift else 0.,
    'final_aperture':  s_lift.position if s_lift else 0.,
    'slip_thresh':     slip_thresh,
    'lift_force_cmd':  lift_force,
    'obj_pos_world':   p_obj.tolist(),
    'grasp_angle_deg': float(ang),
    'grasp_strategy':  strategy,
    'obj_width_mm':    float(obj_w_mm),
    'symmetric':           bool(_sym) if '_sym' in dir() else None,
    'angle_hist_n':        _adv['n_at_proposed']      if _adv else 0,
    'angle_hist_success':  _adv['success_at_proposed'] if _adv else 0,
    'angle_overridden':    bool(_adv['overridden'])   if _adv else False,
    'orientation_reason':  _adv['reason']             if _adv else 'no memory',
    'image_grasp_check':   _grasp_vis if '_grasp_vis' in dir() else None,
    'grasp_check_verdict': _grasp_verdict if '_grasp_verdict' in dir() else 'none',
    'grasp_check_reason':  _grasp_check_reason if '_grasp_check_reason' in dir() else '',
    'pca_angle':            _ang_pca if '_ang_pca' in dir() else None,
    'graspgenx_angle':      _ang_ggx if '_ang_ggx' in dir() else None,
    'angle_arbiter_choice': _ang_arb_choice if '_ang_arb_choice' in dir() else None,
    'angle_correction_deg': float(ang - _ang_pca) if '_ang_pca' in dir() else 0.,
    'angle_corr_confident': bool(_acorr['confident']) if _acorr else False,
    'image_detect':    fp_color,       # RGB image at detection position
    'image_grasp':     snap_grasp,     # RGB image at grasp position (pre-close)
    'image_held':      snap_held,      # RGB image post-lift (shows actual grip)
    'depth_detect':    fp_depth,       # uint16 depth at detection
    'tcp_grasp':       grasp_pose.tolist(),
    'gp_mass_g':       GP.get('mass_g') if 'GP' in dir() and GP else None,
    'gp_mu':           GP.get('mu')     if 'GP' in dir() and GP else None,
    'gp_k':            GP.get('k')      if 'GP' in dir() and GP else None,
    # DAgger / HER training labels
    'sg_fires':            sg_fires,
    'sg_events':           getattr(node, '_sg_events', []).copy(),
    'true_force_label':    true_force_label,
    'force_label_error_n': force_label_error,
    'init_force_pred':     _init_pred,
}

result['grasp_method'] = _GMETHOD if '_GMETHOD' in dir() else 'pca'
if 'CMP_RESULTS' in dir():
    CMP_RESULTS.append(dict(method=result['grasp_method'], held=bool(held),
        final_force=round(float(result['final_force']), 2),
        angle_deg=round(float(ang), 1),
        aperture_mm=round(float(result['final_aperture']), 1),
        sg_fires=result.get('sg_fires', 0)))
    print(f"  [compare] logged {result['grasp_method']}: held={held}  ({len(CMP_RESULTS)} runs)")

if held:
    vla_rec.set_action(home_pick, grip=1.)  # carry object, still holding
    node.move(home_pick, spd=0.08)          # carry object to home
    time.sleep(0.3)
    print('Placing — lowering to table...')
    _place = home_pick.copy()
    _place[2, 3] = grasp_pose[2, 3] + 0.008  # table Z + 8mm clearance
    node.move(_place, spd=0.04)              # slow descent
    time.sleep(0.3)
    vla_rec.set_action(_place, grip=0.)     # release
    node.open_g()
    time.sleep(0.8)                # wait for object to settle before rising
    node.move(home_pick, spd=0.05)           # rise back to home
    time.sleep(0.5); node.spin(4)
    print('Done — object placed.')
else:
    node.open_g()
    node.move(home_pick, spd=0.10)
    print('Dropped — result logged. Adjust params and re-run if needed.')

# ── End VLA episode (buffered; committed by dg_summary once reward is known) ──
_vla_frames = vla_rec.end_episode()
print(f'  [VLA] buffered {_vla_frames} frames @ 10Hz (commit decided by reward gate)')

# ── 10. Final summary ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if grasp_log:
    atts = [str(x['attempt']) for x in grasp_log]
    cols = ['limegreen' if x['status']=='contact' else 'tomato' for x in grasp_log]
    axes[0].bar(atts, [x['force'] for x in grasp_log], color=cols)
    axes[0].axhline(slip_thresh, color='orange', ls='--', lw=2, label=f'thresh {slip_thresh:.2f}N')
    if F_min > 0:
        axes[0].axhline(F_min, color='cyan', ls=':', lw=1.5, label=f'F_min {F_min:.2f}N')
    axes[0].set_ylabel('N'); axes[0].set_title('Grasp Force'); axes[0].legend()
axes[1].scatter(pts_use[:,0], pts_use[:,1], c=pts_use[:,2], cmap='plasma', s=5, alpha=.6)
axes[1].plot(*p_obj[:2], 'r+', ms=18, mew=3, label='centroid')
axes[1].set_xlabel('X m'); axes[1].set_ylabel('Y m')
axes[1].set_title(f'PCD [{src_label}]  angle={ang:.1f}deg'); axes[1].legend()
plt.suptitle(f'Full Pickup — "{ACTIVE}"  -> {"HELD" if held else "DROPPED"}')
plt.tight_layout(); plt.show()
last = grasp_log[-1] if grasp_log else {}
print(f'\nResult: {last.get("status","?")}  ap={last.get("ap",0):.1f}mm  F={last.get("force",0):.3f}N  attempts={len(grasp_log)}')
# No VRAM cycle: SAM3 + GraspGenX coexist (measured ~3 GB headroom on the 8 GB
# card), so both stay warm between grasps — the GraspGenX server is reused, no
# per-grasp reload and no SAM3 restart needed.


In [ ]:
# ── DeliGrasp Summary ────────────────────────────────────────────────────────
# Concise post-grasp record: force, aperture, strategy, memory state, outcome.
# Optionally asks Gemini to describe what it sees in the post-grasp image.

print('\n' + '='*55)
print(f'  Object  : {result.get("object","?")}')
print(f'  Outcome : {"HELD ✓" if result.get("held") else "DROPPED ✗"}')
print(f'  Strategy: {result.get("grasp_strategy","?")}  angle={result.get("grasp_angle_deg",0):.1f}°')
print(f'  Force   : cmd={result.get("lift_force_cmd",0):.1f}N  '
      f'final={result.get("final_force",0):.3f}N  '
      f'thresh={result.get("slip_thresh",0):.2f}N')
print(f'  Aperture: {result.get("final_aperture",0):.1f}mm  '
      f'obj_width={result.get("obj_width_mm",0):.1f}mm')
# Wrist force/torque from the OptoForce FT sensor (degrades gracefully if absent)
_w = getattr(node, 'wrench', None) if 'node' in dir() else None
if _w is not None:
    _f, _t = _w.wrench.force, _w.wrench.torque
    print(f'  Wrist FT: Fz={_f.z:+.2f}N (Fx={_f.x:+.2f} Fy={_f.y:+.2f})  '
          f'Tz={_t.z:+.3f}Nm')
else:
    print(f'  Wrist FT: n/a (FT sensor not publishing — power-cycle OptoForce DAQ)')
print(f'  Slip guard fired: {result.get("sg_fires",0)}x  '
      f'true_force_label={result.get("true_force_label",0):.1f}N')
print(f'  Force pred error: {result.get("force_label_error_n",0):+.2f}N  '
      f'(init={result.get("init_force_pred",0):.1f}N)')
_gp_str = (f'mass={result.get("gp_mass_g","?")}g  '
           f'mu={result.get("gp_mu","?")}  '
           f'k={result.get("gp_k","?")}N/m')
print(f'  DeliGrasp: {_gp_str}')
if 'gm' in dir() and ACTIVE:
    _mp = gm._priors.get(ACTIVE.lower().strip(), {})
    print(f'  Memory  : n={_mp.get("n",0)}  x={_mp.get("x",0):.1f}N  '
          f'P={_mp.get("P",4.0)**0.5:.2f}N std')
print('='*55)

# ── Grasp quality assessment (reward signal for VLA training) ─────────────────
# Sends camera + PCA view to Gemini → quality score 0–1 saved to JSONL.
# This is the reward: did the robot grip the object correctly for its shape?
if 'gc' in dir() and result.get('image_grasp') is not None:
    try:
        from google.genai import types as _gt
        import cv2 as _cv2, json as _json2, pathlib as _pl2

        def _to_jpg_bytes(rgb_arr):
            _, _b = _cv2.imencode('.jpg', _cv2.cvtColor(rgb_arr, _cv2.COLOR_RGB2BGR))
            return _b.tobytes()

        # Orientation history note → lets the reward reason about repeated slips
        _ohist = ''
        if 'gm' in dir() and ACTIVE:
            _h = gm._orient.get(ACTIVE.lower().strip(), [])
            if len(_h) >= 2:
                _hn = sum(1 for x in _h if x['held'])
                _ohist = (f'Orientation history for this object: {_hn}/{len(_h)} past '
                          f'grasps held overall.')

        _parts = []
        _parts.append(_gt.Part.from_bytes(
            data=_to_jpg_bytes(result['image_grasp']), mime_type='image/jpeg'))
        _parts.append('Image 1: Camera view just before gripper closed.')
        if result.get('image_held') is not None:
            _parts.append(_gt.Part.from_bytes(
                data=_to_jpg_bytes(result['image_held']), mime_type='image/jpeg'))
            _parts.append('Image 2: Camera view after lift (actual grip).')
        if '_pca_plot_bytes' in dir() and _pca_plot_bytes:
            _parts.append(_gt.Part.from_bytes(
                data=_pca_plot_bytes, mime_type='image/jpeg'))
            _parts.append('Image 3: PCA top-down point cloud with gripper axis overlay.')
        if result.get('image_grasp_check') is not None:
            _parts.append(_gt.Part.from_bytes(
                data=_to_jpg_bytes(result['image_grasp_check']), mime_type='image/jpeg'))
            _parts.append('Image 4: the gripper CLOSING AXIS (green line) drawn on the '
                          'object, with orange ticks at the finger contact faces.')
        _k = result.get('gp_k')
        if _k is not None and _k >= 400:
            _stiff_note = (f'This object is RIGID (stiffness k={_k:.0f} N/m): it physically '
                           f'CANNOT compress or deform at normal grip forces. Do NOT report '
                           f'deformation/compression for it — any problem is the grasp ANGLE '
                           f'or finger PLACEMENT, not squashing.')
        elif _k is not None:
            _stiff_note = (f'This object is SOFT (stiffness k={_k:.0f} N/m): visible '
                           f'compression or deformation IS possible and should be penalised.')
        else:
            _stiff_note = ''
        _parts.append(
            f'Object: "{result["object"]}"\n'
            f'Strategy: {result["grasp_strategy"]}  Grasp angle: {result["grasp_angle_deg"]:.1f}deg\n'
            f'Outcome: {"HELD" if result["held"] else "DROPPED"}\n'
            f'{_stiff_note}\n{_ohist}\n\n'
            'The GREEN line in the last image is the gripper closing axis. A GOOD grasp '
            'closes across the NARROWEST width onto flat opposing faces, NOT across a '
            'diagonal/corner or the widest span.\n\n'
            'Assess these independently:\n'
            '  A. ANGLE - is the green axis across the narrow faces (good), or '
            'diagonal/corner/widest (bad)?\n'
            '  B. PLACEMENT - are the fingers on the flat body, or on a rim/edge/corner?\n'
            '  C. DEFORMATION - ONLY if the object is soft: is it visibly compressed?\n'
            '  D. STABILITY - is it tilted or about to slip?\n\n'
            'Score 0.0-1.0 (strict - should this grasp be imitated in training?):\n'
            '  1.0 = ideal angle, flat-face contact, stable; (soft) no deformation\n'
            '  0.7-0.9 = good, minor angle/placement imperfection\n'
            '  0.4-0.6 = marginal: clearly off angle (diagonal/corner), edge contact, '
            'or (soft only) visible deformation\n'
            '  0.0-0.3 = poor: bad angle, barely held, or crushing\n\n'
            'Name the PRIMARY problem. Reply in EXACTLY this format:\n'
            'SCORE: <0.0 to 1.0>\n'
            'CATEGORY: <good|marginal|poor>\n'
            'ISSUE: <none|angle|placement|deformation|stability>\n'
            'REASON: <one sentence>'
        )
        _qr = gc.models.generate_content(
            model='gemini-2.5-flash', contents=_parts).text.strip()

        _score = None; _cat = '?'; _reason = ''; _issue = 'none'
        for _line in _qr.split('\n'):
            if _line.startswith('SCORE:'):
                try: _score = float(_line.split(':',1)[1].strip())
                except: pass
            elif _line.startswith('CATEGORY:'): _cat = _line.split(':',1)[1].strip()
            elif _line.startswith('ISSUE:'): _issue = _line.split(':',1)[1].strip().lower()
            elif _line.startswith('REASON:'): _reason = _line.split(':',1)[1].strip()

        print(f'  Quality : {_score:.2f} [{_cat}]  issue={_issue} — {_reason}')

        # Update last JSONL record with quality score
        _lf = _pl2.Path(os.path.expanduser('~/magpie_control/data/grasp_log'))
        _lf = _lf / f'grasps_{_dt.now().strftime("%Y-%m-%d")}.jsonl'
        if _lf.exists():
            _lines = _lf.read_text().splitlines()
            if _lines:
                _last_rec = _json2.loads(_lines[-1])
                _last_rec['grasp_quality'] = _score
                _last_rec['grasp_quality_reason'] = _reason
                _last_rec['grasp_quality_category'] = _cat
                _last_rec['grasp_quality_issue'] = _issue
                _lines[-1] = _json2.dumps(_last_rec, cls=_NpEnc)
                _lf.write_text('\n'.join(_lines) + '\n')
                print(f'  JSONL updated with quality score.')

        # Reward feedback into the orientation memory + next-attempt advice
        if 'gm' in dir() and ACTIVE:
            if _score is not None:
                gm.set_last_quality(ACTIVE, _score)
            _next = gm.get_angle_advice(
                ACTIVE, result.get('grasp_angle_deg', 0.),
                symmetric=bool(result.get('symmetric', False)))
            _np_, _rt_ = _next['n_at_proposed'], (_next['rate_at_proposed'] or 0.)
            if _np_ >= 2 and _rt_ < 0.5:
                print(f'  ⟳ Orientation advice: {_next["reason"]}')
                if _next['overridden']:
                    print(f'     next grasp will try ~{_next["recommend_angle"]:.1f}°')
    except Exception as _e:
        print(f'  Quality : [skipped: {_e}]')

# ── Commit the VLA episode through the reward gate ──────────────────────────
# Saved to the trainable LeRobot dataset ONLY if held AND quality >= 0.6.
# Failed/low-quality grasps are logged to attempts_log but never imitated.
if 'vla_rec' in dir():
    _q = _score if '_score' in dir() else None
    _centering_bad = result.get('centering_flag') == 'unreliable'
    if _centering_bad:
        # Pre-descent centering couldn't be verified/corrected — don't trust this
        # grasp even if it held. Force it below the reward-gate threshold so it
        # never enters the trainable LeRobot set; it still logs to attempts_log.
        _q = 0.0
        print('  [reward gate] centering unreliable -> forcing quality=0.0, '
              'EXCLUDED from training set (still logged to attempts)')
    _vla_saved = vla_rec.commit(
        success=bool(result.get('held')) and not _centering_bad, reward=_q, threshold=0.6,
        extra={'object': result.get('object'),
               'grasp_angle_deg': result.get('grasp_angle_deg'),
               'quality_issue': _issue if '_issue' in dir() else None,
               'centering_flag': result.get('centering_flag'),
               'sg_fires': result.get('sg_fires')})
    if _vla_saved:
        # Flush the episode to disk immediately so the file is valid even if kernel dies
        vla_rec.flush_episode()
        # Open Rerun to replay the saved episode
        import subprocess as _sp, pathlib as _pp
        _ds_root = _pp.Path('~/magpie_control/data/lerobot_magpie').expanduser()
        _ep_files = sorted(_ds_root.glob('data/chunk-*/file-*.parquet'))
        _ep_idx = max(0, len(_ep_files) - 1)
        _rr_cmd = ['python3', '-m', 'lerobot.scripts.lerobot_dataset_viz',
                   '--repo-id', 'magpie/grasp', '--root', str(_ds_root),
                   '--episode-index', str(_ep_idx)]
        _sp.Popen(_rr_cmd)
        print(f'  [VLA] Rerun viewer opening for episode {_ep_idx} ...')

# Optional: one-sentence grip description
if 'gc' in dir() and result.get('image_grasp') is not None and False:  # disabled — quality above covers this
    try:
        from google.genai import types as _gt
        import cv2 as _cv2
        _, _buf = _cv2.imencode('.jpg',
            _cv2.cvtColor(result['image_grasp'], _cv2.COLOR_RGB2BGR))
        _summary = gc.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                _gt.Part.from_bytes(data=_buf.tobytes(), mime_type='image/jpeg'),
                'One sentence: describe what the gripper is holding and its grip quality.'
            ]).text.strip()
        print(f'  VLM     : {_summary}')
    except Exception as _e:
        print(f'  VLM     : [skipped: {_e}]')
